# Solar Panel Dust Classification — Preprocessing & Feature Extraction

Produces `solar_features.pkl` containing EfficientNetB0 embeddings (1280-d),
stratified splits, and a fitted StandardScaler — ready for downstream training.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "numpy", "pandas", "scikit-learn", "tensorflow"
])
print("Dependencies ready.")

In [ ]:
# Cell 2 — Imports
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

print(f"TensorFlow {tf.__version__}  |  NumPy {np.__version__}  |  pandas {pd.__version__}")

In [ ]:
# Cell 3 — Dataset scan
DATASET_ROOT = Path("/kaggle/input/datasets/mabubakar29/solardataset/processed")
VALID_EXTS   = {".jpg", ".jpeg", ".png"}

CLASS_MAP = {"Clean": 0, "Heavy_Dust": 1, "Low_Dust": 2}

records = []
for class_name, label in CLASS_MAP.items():
    class_dir = DATASET_ROOT / class_name
    if not class_dir.is_dir():
        raise FileNotFoundError(f"Expected class folder not found: {class_dir}")
    for p in class_dir.iterdir():
        if p.suffix.lower() in VALID_EXTS:
            records.append({"path": str(p), "label": label, "class": class_name})

df = pd.DataFrame(records)
print(f"Total images found: {len(df)}")
print(df.groupby("class")["path"].count().to_string())

In [ ]:
# Cell 4 — Balance classes via oversampling
majority_count = df["class"].value_counts().max()

balanced_parts = []
for class_name in CLASS_MAP:
    subset = df[df["class"] == class_name]
    resampled = resample(
        subset,
        replace=True,
        n_samples=majority_count,
        random_state=42,
    )
    balanced_parts.append(resampled)

df_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("Before balancing:")
print(df.groupby("class")["path"].count().to_string())
print(f"\nAfter balancing (majority_count={majority_count} per class):")
print(df_balanced.groupby("class")["path"].count().to_string())
print(f"Total balanced: {len(df_balanced)}")

In [ ]:
# Cell 5 — Build frozen EfficientNetB0 feature extractor
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base.trainable = False

output  = GlobalAveragePooling2D()(base.output)
feature_extractor = Model(inputs=base.input, outputs=output)

print(f"Feature extractor output shape: {feature_extractor.output_shape}  (expected (None, 1280))")

In [ ]:
# Cell 6 — Extract features batch by batch
def load_and_preprocess(path: str) -> np.ndarray:
    """Load a single image, resize, and apply EfficientNet preprocessing."""
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)           # float32 HxWxC
    return preprocess_input(arr)                      # EfficientNet expects this


paths  = df_balanced["path"].tolist()
labels = df_balanced["label"].to_numpy()
n      = len(paths)

feature_batches = []

for batch_idx in range(0, n, BATCH_SIZE):
    batch_paths = paths[batch_idx : batch_idx + BATCH_SIZE]
    batch_imgs  = np.stack([load_and_preprocess(p) for p in batch_paths])  # (B, 224, 224, 3)
    batch_feats = feature_extractor.predict(batch_imgs, verbose=0)          # (B, 1280)
    feature_batches.append(batch_feats)

    batch_num = batch_idx // BATCH_SIZE + 1
    if batch_num % 10 == 0 or (batch_idx + BATCH_SIZE) >= n:
        done = min(batch_idx + BATCH_SIZE, n)
        print(f"  Batch {batch_num:>4d} — processed {done}/{n} images")

X_all = np.vstack(feature_batches)   # (N, 1280)
y_all = labels

print(f"\nFeature matrix shape: {X_all.shape}  labels shape: {y_all.shape}")

In [ ]:
# Cell 7 — Stratified train / val / test split  (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all,
    test_size=0.30,
    stratify=y_all,
    random_state=42,
)

# Split the remaining 30 % equally into val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

print(f"Train : {X_train.shape[0]:>5d} samples")
print(f"Val   : {X_val.shape[0]:>5d} samples")
print(f"Test  : {X_test.shape[0]:>5d} samples")

In [ ]:
# Cell 8 — Fit StandardScaler on train only, transform val and test
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Scaler fitted on {X_train.shape[0]} training samples.")
print(f"Train mean (first 5 features): {X_train.mean(axis=0)[:5].round(4)}")
print(f"Train std  (first 5 features): {X_train.std(axis=0)[:5].round(4)}")

In [ ]:
# Cell 9 — Save to pkl
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "solar_features.pkl"

payload = {
    "X_train":  X_train,
    "X_val":    X_val,
    "X_test":   X_test,
    "y_train":  y_train,
    "y_val":    y_val,
    "y_test":   y_test,
    "classes":  ["Clean", "Heavy_Dust", "Low_Dust"],
    "scaler":   scaler,
}

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"Saved to {OUTPUT_PATH}  ({size_mb:.1f} MB)")

In [ ]:
# Cell 10 — Verify: reload and print shapes
with open(OUTPUT_PATH, "rb") as f:
    loaded = pickle.load(f)

print("Reloaded payload keys:", list(loaded.keys()))
print()
for key in ("X_train", "X_val", "X_test", "y_train", "y_val", "y_test"):
    arr = loaded[key]
    print(f"  {key:<10s}  shape={arr.shape}  dtype={arr.dtype}")

print(f"\n  classes : {loaded['classes']}")
print(f"  scaler  : {loaded['scaler']}")
print()

# Sanity checks
assert loaded["X_train"].shape[1] == 1280, "Expected 1280 features"
assert loaded["X_train"].shape[0] == loaded["y_train"].shape[0]
assert loaded["X_val"].shape[0]   == loaded["y_val"].shape[0]
assert loaded["X_test"].shape[0]  == loaded["y_test"].shape[0]
print("All sanity checks passed.")